# Tmp Data Full API Tutorial

This notebook mirrors the full validator flow on `tmp_data`.

The default OSS production path is a disk-backed local collection root with exact Triton MaxSim in FP16 for late-interaction and multimodal retrieval. Optional quantization profiles are useful for benchmarking and memory reduction, but they are not the default story.

- render pages and sheets into image assets
- attach ontology-derived enrichment from `dataset_di_825cbaae40335bc4265a3726.json`
- prepare a shared evaluation bundle with real ColLFM2 embeddings when available
- create reference API collections with and without ontology enrichment
- exercise CRUD and compare MaxSim, quantization, solver, and ontology lanes


In [ ]:
from __future__ import annotations

import json
import os
import sys
import tempfile
from pathlib import Path

from fastapi.testclient import TestClient

repo_root = Path.cwd()
if not (repo_root / "scripts" / "full_feature_validation.py").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.full_feature_validation import (
    bootstrap_native_packages,
    enumerate_source_documents,
    load_ontology_entities,
    prepare_evaluation_bundle,
    render_documents,
    resolve_ontology_fixture,
    run_maxsim_validation,
    run_ontology_variant_evaluation,
    run_quantization_validation,
    run_solver_lane,
)
from voyager_index.server import create_app

# Override with VOYAGER_TMP_DATA_DIR when using a shared corpus outside the repo.
tmp_data_dir = Path(os.environ.get("VOYAGER_TMP_DATA_DIR", str(repo_root.parent / "tmp_data")))
ontology_fixture = resolve_ontology_fixture(tmp_data_dir, None)
output_dir = repo_root / "validation-reports" / "notebook-tmp-data-demo"
rendered_dir = output_dir / "rendered-corpus"
output_dir.mkdir(parents=True, exist_ok=True)

print({
    "repo_root": str(repo_root),
    "tmp_data_dir": str(tmp_data_dir),
    "ontology_fixture": str(ontology_fixture),
    "output_dir": str(output_dir),
})


In [ ]:
inventory = enumerate_source_documents(tmp_data_dir, ontology_fixture)
rendered_manifest = render_documents(inventory["documents"], rendered_dir)
entities = load_ontology_entities(ontology_fixture, limit=1024)
evaluation_bundle = prepare_evaluation_bundle(rendered_manifest.get("rendered", []), entities, output_dir)

summary = {
    "documents": len(inventory["documents"]),
    "rendered_pages_or_sheets": len(rendered_manifest.get("rendered", [])),
    "records": evaluation_bundle["corpus_summary"]["record_count"],
    "queries": evaluation_bundle["corpus_summary"]["query_count"],
    "records_with_ontology_matches": evaluation_bundle["corpus_summary"]["records_with_ontology_matches"],
    "embedding_source": evaluation_bundle["embedding_summary"]["embedding_source"],
}
summary


In [ ]:
records = evaluation_bundle["records"]
queries = evaluation_bundle["queries"]
top_k = min(3, len(records))
dense_base_points = [
    {
        "id": record["id"],
        "vector": record["dense_vector"].tolist(),
        "payload": {
            "text": record["base_text"],
            "document_id": record["document_id"],
            "page_number": record["page"],
            "ontology_terms": record["ontology_terms"],
        },
    }
    for record in records
]
dense_ontology_points = [
    {
        "id": record["id"],
        "vector": record["dense_vector"].tolist(),
        "payload": {
            "text": record["enriched_text"],
            "document_id": record["document_id"],
            "page_number": record["page"],
            "ontology_terms": record["ontology_terms"],
        },
    }
    for record in records
]
np = __import__("numpy")
max_tokens = max(record["multivector"].shape[0] for record in records)
multivector_points = []
for record in records:
    padded = record["multivector"]
    if padded.shape[0] != max_tokens:
        target = np.zeros((max_tokens, padded.shape[1]), dtype=np.float32)
        target[: padded.shape[0]] = padded
        padded = target
    multivector_points.append(
        {
            "id": record["id"],
            "vectors": padded.tolist(),
            "payload": {
                "text": record["enriched_text"],
                "document_id": record["document_id"],
                "page_number": record["page"],
            },
        }
    )
query = queries[0]
query_multivector = query["multivector"]
if query_multivector.shape[0] != max_tokens:
    target = np.zeros((max_tokens, query_multivector.shape[1]), dtype=np.float32)
    target[: query_multivector.shape[0]] = query_multivector
    query_multivector = target

with tempfile.TemporaryDirectory() as tmpdir:
    app = create_app(index_path=str(Path(tmpdir) / "api"))
    with TestClient(app) as client:
        client.post("/collections/dense-base", json={"dimension": len(dense_base_points[0]["vector"]), "kind": "dense"})
        client.post("/collections/dense-ontology", json={"dimension": len(dense_ontology_points[0]["vector"]), "kind": "dense"})
        client.post("/collections/li", json={"dimension": len(multivector_points[0]["vectors"][0]), "kind": "late_interaction"})
        client.post("/collections/mm", json={"dimension": len(multivector_points[0]["vectors"][0]), "kind": "multimodal"})
        client.post("/collections/dense-base/points", json={"points": dense_base_points})
        client.post("/collections/dense-ontology/points", json={"points": dense_ontology_points})
        client.post("/collections/li/points", json={"points": multivector_points})
        client.post("/collections/mm/points", json={"points": multivector_points})

        without_ontology = client.post(
            "/collections/dense-base/search",
            json={"vector": query["dense_vector"].tolist(), "query_text": query["text"], "top_k": top_k},
        ).json()
        with_ontology = client.post(
            "/collections/dense-ontology/search",
            json={"vector": query["dense_vector"].tolist(), "query_text": query["text"], "top_k": top_k},
        ).json()
        multimodal = client.post(
            "/collections/mm/search",
            json={"vectors": query_multivector.tolist(), "top_k": top_k, "with_vector": True},
        ).json()

        updated = dense_base_points[0] | {"payload": dense_base_points[0]["payload"] | {"text": dense_base_points[0]["payload"]["text"] + " updated"}}
        client.post("/collections/dense-base/points", json={"points": [updated]})
        client.request("DELETE", "/collections/dense-base/points", json={"ids": [dense_base_points[-1]["id"]]})

        comparison = {
            "query": query["text"],
            "without_ontology_top_ids": [item["id"] for item in without_ontology["results"]],
            "with_ontology_top_ids": [item["id"] for item in with_ontology["results"]],
            "multimodal_top_ids": [item["id"] for item in multimodal["results"]],
        }

comparison


In [ ]:
native_packages = bootstrap_native_packages(output_dir, bootstrap=False)

lane_summaries = {
    "maxsim": run_maxsim_validation(output_dir, evaluation_bundle),
    "quantization": run_quantization_validation(output_dir, evaluation_bundle),
    "solver": run_solver_lane(output_dir, native_packages, evaluation_bundle),
    "ontology_variant": run_ontology_variant_evaluation(output_dir, native_packages, evaluation_bundle),
}

{key: value["status"] for key, value in lane_summaries.items()}


In [ ]:
compact_report = {
    key: {
        "status": value["status"],
        "reason": value.get("reason"),
        "path": value.get("path"),
    }
    for key, value in lane_summaries.items()
}
print(json.dumps(compact_report, indent=2))


## Notes

- The notebook uses the same helper functions as `scripts/full_feature_validation.py`, so its outputs should line up with the validator report.
- Collections persist under the configured local root; in-memory tensors and VRAM caches are runtime accelerators rather than a separate storage backend.
- If `sauerkrautlm-colpali` or CUDA is unavailable, the bundle falls back to synthetic embeddings so the API flow still runs.
- INT8 and RoQ4 are best treated as optional profiles to compare against the exact FP16 default, not as hidden defaults.
- If `latence_solver` is not installed, the solver lane will report a truthful skip instead of pretending the backend is active.
- The OSS solver lane defaults to the Rust CPU fallback unless an accelerated backend is built and available; deeper Voyager-only backends live in the separate `latence-voyager` repo.
